In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import GATConv
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [7]:
# --- CONFIGURATION ---
DATASET_FILE = "/content/drive/MyDrive/BWAF-Net-Training/Final_Aligned_Dataset.csv"
GRAPH_FILE = "/content/drive/MyDrive/BWAF-Net-Training/graph_data.pt"
OUTPUT_DIR = "/content/drive/MyDrive/BWAF-Net-Training/results_bwaf/"
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "best_bwaf_model.pth")
BATCH_SIZE = 16
LR = 0.0005
EPOCHS = 15
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [8]:
# --- MODEL DEFINITION (BWAF-Net) ---
class BWAF_Net(nn.Module):
    def __init__(self, num_genes, num_tf_features, d_model=64, prior_dim=11):
        super(BWAF_Net, self).__init__()

        # 1. Sequence Branch
        self.embedding = nn.Embedding(5, d_model, padding_idx=4) # A=0,C=1,G=2,T=3,N=4
        self.pos_encoder = nn.Parameter(torch.randn(1, 2000, d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=4, batch_first=True, dim_feedforward=256)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.seq_proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # 2. Graph Branch (Online GAT)
        self.gat = GATConv(num_tf_features, d_model, heads=1)
        self.register_buffer('edge_index', None)
        self.register_buffer('node_features', None)

        # 3. BWAF Fusion (Knowledge Controller)
        self.seq_gate = nn.Sequential(nn.Linear(prior_dim, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())
        self.graph_gate = nn.Sequential(nn.Linear(prior_dim, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())

        # 4. Classifier
        # Input: fused(64) + priors(11) = 75
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model + prior_dim),
            nn.Linear(d_model + prior_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def set_graph_data(self, x, edge_index):
        self.edge_index = edge_index
        self.node_features = x

    def forward(self, seq, priors, gene_idx):
        # A. Sequence
        # Mask padding (N=4)
        pad_mask = (seq == 4)
        x_seq = self.embedding(seq) + self.pos_encoder
        x_seq = self.transformer(x_seq, src_key_padding_mask=pad_mask)
        # Global Average Pooling (ignoring padding)
        # Simple mean is risky if lots of padding, but for promoters (mostly full), it's fine.
        # Let's do a masked mean for rigor.
        mask = (~pad_mask).unsqueeze(-1).float()
        x_seq = (x_seq * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        h_seq = self.seq_proj(x_seq)

        # B. Graph (Run on ALL nodes, select BATCH)
        # This updates GAT weights based on batch loss
        all_nodes = F.elu(self.gat(self.node_features, self.edge_index))
        h_graph = all_nodes[gene_idx]

        # C. Fusion
        w_seq = self.seq_gate(priors)
        w_graph = self.graph_gate(priors)
        h_fused = (w_seq * h_seq) + (w_graph * h_graph)

        # D. Output
        h_final = torch.cat([h_fused, priors], dim=1)
        return self.classifier(h_final)

# --- DATASET ---
class GenomicDataset(Dataset):
    def __init__(self, df, gene_to_idx):
        self.seqs = df['sequence'].values
        self.labels = df['label'].values
        # Map gene string IDs to integer indices for graph lookup
        self.gene_idxs = [gene_to_idx.get(g, 0) for g in df['gene_id'].values]

        # Priors: Select 11 cols
        prior_cols = ["TATA_Box", "CAAT_Box", "GC_Box", "BRE", "MRE", "PPE",
                      "Octamer", "Sp1", "E_Box", "RFX", "CpG_Count"]
        self.priors = np.log1p(df[prior_cols].values.astype(np.float32))

        self.dna_map = {'A':0, 'C':1, 'G':2, 'T':3, 'N':4}

    def __len__(self): return len(self.seqs)

    def __getitem__(self, i):
        # Convert seq string to int list
        seq = [self.dna_map.get(s, 4) for s in self.seqs[i]]
        return (torch.tensor(seq, dtype=torch.long),
                torch.tensor(self.priors[i], dtype=torch.float32),
                torch.tensor(self.gene_idxs[i], dtype=torch.long),
                torch.tensor(self.labels[i], dtype=torch.float32))

# --- PLOTTING UTILS ---
def plot_loss(train_losses, val_losses):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.title('Training vs Validation Loss')
    plt.legend()
    plt.savefig(os.path.join(OUTPUT_DIR, "loss_curve.png"))
    plt.close()

def plot_cm(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"))
    plt.close()

# --- TRAINING LOOP ---
def train():
    print(f"--- STARTING TRAINING (Device: {DEVICE}) ---")

    # Load Data
    df = pd.read_csv(DATASET_FILE)
    print(f"Loaded {len(df)} samples.")

    # Load Graph
    print("Loading Graph Data...")
    graph = torch.load(GRAPH_FILE)
    gene_to_idx = graph['gene_to_idx']

    # Split Data
    train_df = df[df['partition'] == 'train']
    val_df = df[df['partition'] == 'val']
    test_df = df[df['partition'] == 'test']

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

    # Loaders
    train_loader = DataLoader(GenomicDataset(train_df, gene_to_idx), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(GenomicDataset(val_df, gene_to_idx), batch_size=BATCH_SIZE)
    test_loader = DataLoader(GenomicDataset(test_df, gene_to_idx), batch_size=BATCH_SIZE)

    # Init Model
    num_genes = graph['x'].shape[0]
    num_tfs = graph['x'].shape[1]
    model = BWAF_Net(num_genes, num_tfs).to(DEVICE)

    # **CRITICAL**: Load graph structure into model buffers
    model.set_graph_data(graph['x'].to(DEVICE), graph['edge_index'].to(DEVICE))

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2)

    best_val_loss = float('inf')
    train_losses, val_losses = [], []

    for ep in range(EPOCHS):
        model.train()
        epoch_loss = 0

        loop = tqdm(train_loader, desc=f"Ep {ep+1}/{EPOCHS}")
        for seq, prior, gidx, y in loop:
            seq, prior, gidx, y = seq.to(DEVICE), prior.to(DEVICE), gidx.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(seq, prior, gidx).squeeze()
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        avg_train_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation
        model.eval()
        val_loss = 0
        preds, targs = [], []
        with torch.no_grad():
            for seq, prior, gidx, y in val_loader:
                seq, prior, gidx, y = seq.to(DEVICE), prior.to(DEVICE), gidx.to(DEVICE), y.to(DEVICE)
                logits = model(seq, prior, gidx).squeeze()
                val_loss += criterion(logits, y).item()

                preds.extend(torch.sigmoid(logits).cpu().numpy())
                targs.extend(y.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        # Metrics
        preds_bin = np.array(preds) > 0.5
        acc = accuracy_score(targs, preds_bin)
        auc = roc_auc_score(targs, preds)

        print(f"Ep {ep+1} | Val Loss: {avg_val_loss:.4f} | Acc: {acc:.4f} | AUC: {auc:.4f}")

        scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(">>> Saved Best Model")

    # --- FINAL TEST ---
    print("\n--- RUNNING FINAL TEST EVALUATION ---")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    model.eval()

    preds, targs, probs = [], [], []
    with torch.no_grad():
        for seq, prior, gidx, y in tqdm(test_loader, desc="Testing"):
            seq, prior, gidx, y = seq.to(DEVICE), prior.to(DEVICE), gidx.to(DEVICE), y.to(DEVICE)
            logits = model(seq, prior, gidx).squeeze()
            prob = torch.sigmoid(logits)

            probs.extend(prob.cpu().numpy())
            preds.extend((prob > 0.5).float().cpu().numpy())
            targs.extend(y.cpu().numpy())

    # Metrics
    acc = accuracy_score(targs, preds)
    auc = roc_auc_score(targs, probs)
    prec = precision_score(targs, preds)
    rec = recall_score(targs, preds)
    f1 = f1_score(targs, preds)

    print("\n=== TEST RESULTS ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"AUC-ROC:   {auc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")

    # Plots
    plot_loss(train_losses, val_losses)
    plot_cm(targs, preds)
    print(f"Results saved to {OUTPUT_DIR}")

if __name__ == "__main__":
    train()

--- STARTING TRAINING (Device: cuda) ---
Loaded 35086 samples.
Loading Graph Data...
Train: 23782 | Val: 5134 | Test: 6170


Ep 1/15: 100%|██████████| 1486/1486 [07:13<00:00,  3.43it/s, loss=0.62]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Ep 1 | Val Loss: 0.5603 | Acc: 0.7178 | AUC: 0.8204
>>> Saved Best Model


Ep 2/15: 100%|██████████| 1486/1486 [07:19<00:00,  3.38it/s, loss=0.67]


Ep 2 | Val Loss: 0.5137 | Acc: 0.7513 | AUC: 0.8255
>>> Saved Best Model


...

Ep 14 | Val Loss: 0.2753 | Acc: 0.8820 | AUC: 0.9529


Ep 15/15: 100%|██████████| 1486/1486 [07:19<00:00,  3.38it/s, loss=0.336]


Ep 15 | Val Loss: 0.2616 | Acc: 0.8909 | AUC: 0.9561

--- RUNNING FINAL TEST EVALUATION ---


Testing: 100%|██████████| 386/386 [00:33<00:00, 11.58it/s]



=== TEST RESULTS ===
Accuracy:  0.8893
AUC-ROC:   0.9556
Precision: 0.9170
Recall:    0.8561
F1 Score:  0.8855
Results saved to /content/drive/MyDrive/BWAF-Net-Training/results_bwaf/
